In [1]:
import datetime
import numpy as np
import pandas as pd

# 1. Setup parameter dasar
np.random.seed(42)
num_unique_orders = 1800 

# ==========================================================
# TABEL DIMENSI 1: STORES (Kotor - Typo & Case Inconsistency)
# ==========================================================
stores_data = {
    "store_id": ["ST-001", "ST-002", "ST-003", "ST-004", "ST-005"],
    "region": ["Jawa", "Sumatera ", "kalimantan", "Sulawesi", "bali & nusa tenggara"]
}
df_stores = pd.DataFrame(stores_data)
region_weights = [0.45, 0.23, 0.16, 0.10, 0.06]

# ==========================================================
# TABEL DIMENSI 2: CUSTOMERS (Kotor - Missing Values & Case Inconsistency)
# ==========================================================
num_customers = 150
segments = ["Consumer", "Corporate", "Home Office", "CONSUMER", "home office ", None]
segment_p = [0.40, 0.30, 0.10, 0.10, 0.05, 0.05]

customers_list = []
for cust_idx in range(1, num_customers + 1):
    c_id = f"CUST-{cust_idx:03d}"
    segment = np.random.choice(segments, p=segment_p)
    customers_list.append({
        "customer_id": c_id,
        "customer_name": f"Pelanggan {cust_idx}",
        "segment": segment
    })
df_customers = pd.DataFrame(customers_list)

# ==========================================================
# TABEL DIMENSI 3: PRODUCTS (Master Rentang Harga Realistis)
# ==========================================================
product_catalog = {
    "Electronics": {
        "Smartphone Polytron X": {"price_range": (2000000, 4500000), "base_margin": 0.26},
        "Laptop Axioo MyBook": {"price_range": (4500000, 7500000), "base_margin": 0.32},
        "Tablet Advan Sketsa": {"price_range": (2000000, 3500000), "base_margin": 0.32},
        "Smart TV Aqua 32 Inch": {"price_range": (1800000, 3000000), "base_margin": 0.28},
        "Monitor LED Advance 24": {"price_range": (1200000, 1800000), "base_margin": 0.28},
        "Speaker Aktif Polytron": {"price_range": (800000, 1500000), "base_margin": 0.27},
    },
    "Furniture": {
        "Sofa Minimalis Jepara": {"price_range": (3500000, 6500000), "base_margin": 0.24},
        "Meja Makan Kayu Jati": {"price_range": (2500000, 5000000), "base_margin": 0.22},
        "Lemari Pakaian Sliding": {"price_range": (1500000, 3000000), "base_margin": 0.25},
        "Kursi Kerja Ergonomis": {"price_range": (800000, 1500000), "base_margin": 0.24},
        "Rak Buku Kayu Minimalis": {"price_range": (500000, 1200000), "base_margin": 0.23},
        "Springbed Olympic No 2": {"price_range": (2000000, 4000000), "base_margin": 0.25},
    },
    "Home Appliances": {
        "Rice Cooker Cosmos": {"price_range": (300000, 600000), "base_margin": 0.27},
        "Dispenser Denpoo": {"price_range": (800000, 1500000), "base_margin": 0.27},
        "Blender Miyako 3 in 1": {"price_range": (250000, 450000), "base_margin": 0.26},
        "Kipas Angin Maspion": {"price_range": (200000, 500000), "base_margin": 0.27},
        "Air Fryer Mito Digifry": {"price_range": (800000, 1300000), "base_margin": 0.28},
    },
    "Clothing": {
        "Kemeja Batik Premium": {"price_range": (250000, 600000), "base_margin": 0.30},
        "Hijab Voal Eksklusif": {"price_range": (50000, 150000), "base_margin": 0.35},
        "Kaos Polos Cotton Combed": {"price_range": (45000, 95000), "base_margin": 0.40},
        "Jaket Parka Erigo": {"price_range": (180000, 350000), "base_margin": 0.32},
        "Celana Chino Lokal": {"price_range": (120000, 250000), "base_margin": 0.30},
    },
    "Sports": {
        "Sepeda Lipat Element": {"price_range": (2000000, 4500000), "base_margin": 0.25},
        "Matras Yoga Speeds": {"price_range": (80000, 180000), "base_margin": 0.35},
        "Raket Badminton Specs": {"price_range": (250000, 550000), "base_margin": 0.30},
        "Sepatu Lari Mills": {"price_range": (300000, 600000), "base_margin": 0.28},
        "Jersey Timnas Mills": {"price_range": (150000, 380000), "base_margin": 0.33},
    },
}

dirty_categories = {
    "Electronics": "ELECTRONICS",
    "Furniture": "Furniture ",
    "Home Appliances": "home appliances",
    "Clothing": "Clothing",
    "Sports": "Sports"
}

products_list = []
prod_id_counter = 1
prod_map = {}

for cat, prods in product_catalog.items():
    for p_name, info in prods.items():
        p_id = f"PRD-{prod_id_counter:03d}"
        products_list.append({
            "product_id": p_id,
            "product_name": p_name,
            "category": dirty_categories[cat]
        })
        # Simpan informasi detail ke map untuk referensi pembuatan data order
        prod_map[p_id] = {
            "name": p_name, 
            "range": info["price_range"], 
            "margin": info["base_margin"]
        }
        prod_id_counter += 1

df_products = pd.DataFrame(products_list)
all_product_ids = df_products["product_id"].tolist()

# ==========================================================
# TABEL FAKTA: ORDERS (Januari 2023 - Juni 2026)
# ==========================================================
orders_list = []

start_date = datetime.date(2023, 1, 1)
end_date = datetime.date(2026, 6, 30)
days_between_dates = (end_date - start_date).days

# Distribusi kategori transaksi (Elektronik & Furnitur biasanya punya volume item lebih sedikit tapi nominal besar)
category_pool = []
for p_id, info in prod_map.items():
    category_pool.append(p_id)

for idx in range(1, num_unique_orders + 1):
    # --- LEVEL HEADER TRANSAKSI ---
    random_number_of_days = np.random.randint(0, days_between_dates + 1)
    order_date = start_date + datetime.timedelta(days=random_number_of_days)
    
    year = order_date.year
    order_id = f"ID-{year}-{idx:05d}"
    store_id = np.random.choice(df_stores["store_id"], p=region_weights)
    customer_id = np.random.choice(df_customers["customer_id"])
    
    # Menentukan jumlah produk unik dalam 1 transaksi
    num_items_in_order = np.random.choice([1, 2, 3, 4], p=[0.60, 0.25, 0.12, 0.03])
    chosen_products = np.random.choice(all_product_ids, size=num_items_in_order, replace=False)
    
    # --- LEVEL DETAIL TRANSAKSI ---
    for product_id in chosen_products:
        rand_check = np.random.rand()
        p_info = prod_map[product_id]
        
        # 1. Handling Quantity & Data Kotor Quantity
        if rand_check < 0.02:    # 2% Quantity NaN
            quantity = None
        elif rand_check < 0.04:  # 2% Quantity minus
            quantity = -1
        else:
            quantity = int(np.random.choice([1, 2, 3, 5], p=[0.70, 0.18, 0.09, 0.03]))

        # 2. Handling Unit Price & Cost secara Realistis sesuai rentang produk asli
        if rand_check > 0.98:    # 2% Harga NaN
            unit_price = None
            cost = round(np.mean(p_info["range"]) * (1 - p_info["margin"]), -3)
        elif rand_check > 0.96:  # 2% Harga nol
            unit_price = 0
            cost = 0
        else:
            # Ambil acak dari rentang asli produknya sendiri
            base_price = np.random.uniform(p_info["range"][0], p_info["range"][1])
            
            # Efek inflasi/kenaikan harga berkala dari 2023 ke 2026 (~5% per tahun)
            inflation = 1.0 + (year - 2023) * 0.05
            unit_price = round(base_price * inflation, -3)
            
            # Hitung Cost (HPP) berdasarkan margin spesifik produk tersebut
            exact_margin = p_info["margin"] + np.random.uniform(-0.02, 0.02)
            cost = round(unit_price * (1 - exact_margin), -3)

        diskon = round(np.random.choice([0.0, 0.05, 0.10, 0.15], p=[0.80, 0.10, 0.07, 0.03]), 2)

        orders_list.append({
            "order_id": order_id,
            "order_date": order_date,
            "customer_id": customer_id,
            "store_id": store_id,
            "product_id": product_id,
            "quantity": quantity,
            "unit_price": unit_price,
            "discount": diskon,
            "cost": cost
        })

df_orders = pd.DataFrame(orders_list)

# Tambahkan duplikasi data baris secara utuh (~50 baris)
df_duplicates = df_orders.sample(n=50, random_state=42)
df_orders = pd.concat([df_orders, df_duplicates], ignore_index=True)
df_orders = df_orders.sort_values(by=["order_date", "order_id"]).reset_index(drop=True)

# ==========================================================
# 5. EKSKUSI SIMPAN
# ==========================================================
df_orders.to_csv("orders.csv", index=False)
df_products.to_csv("product.csv", index=False)
df_stores.to_csv("store.csv", index=False)
df_customers.to_csv("customer.csv", index=False)

print("Sukses! Dataset super realistis berhasil di-generate.")
print(f"Sekarang harga kaos berkisar puluhan ribu, sedangkan laptop jutaan rupiah.")

Sukses! Dataset super realistis berhasil di-generate.
Sekarang harga kaos berkisar puluhan ribu, sedangkan laptop jutaan rupiah.
